In [4]:
# PLE_ResNet_fast_final_v5_Aligned.py
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                               recall_score, f1_score, mean_absolute_error)
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, Conv2D, BatchNormalization, MaxPooling1D, MaxPooling2D,
                                     GlobalAveragePooling1D, GlobalAveragePooling2D, Dense, Reshape, Layer,
                                     Concatenate, Lambda, Add, Multiply, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy, SparseCategoricalCrossentropy, MeanSquaredError

# ---------------------- 1. 物理参数定义 ----------------------
TYPE_NAMES = ["SingleTone", "Chirp", "Pulse", "HoppingJam", "NoiseFM", "NoiseAM", "Comb", "Mixed", "Normal"]
TRUTH_MAP = {
    0: [500, 0], 1: [60000, 0], 2: [45000, 0], 3: [85000, 0],
    4: [70000, 0], 5: [35000, 0], 6: [65000, 0], 7: [80000, 0], 8: [25000, 0]
}
FS = 200000

# ---------------------- 2. 自定义组件 (保持不动) ----------------------
class PLELayer(Layer):
    def __init__(self, num_tasks, num_shared_experts, num_task_experts, expert_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_tasks, self.num_shared_experts, self.num_task_experts, self.expert_dim = num_tasks, num_shared_experts, num_task_experts, expert_dim
    def build(self, input_shape):
        self.shared_experts = [Dense(self.expert_dim, activation='relu') for _ in range(self.num_shared_experts)]
        self.task_experts = [[Dense(self.expert_dim, activation='relu') for _ in range(self.num_task_experts)] for _ in range(self.num_tasks)]
        self.gates = [Dense(self.num_shared_experts + self.num_task_experts, activation='softmax') for _ in range(self.num_tasks)]
    def call(self, inputs):
        shared_outputs = [ex(inputs) for ex in self.shared_experts]
        final_outputs = []
        for t in range(self.num_tasks):
            task_outputs = [ex(inputs) for ex in self.task_experts[t]]
            all_ex = tf.stack(shared_outputs + task_outputs, axis=1)
            gate = tf.expand_dims(self.gates[t](inputs), axis=-1)
            final_outputs.append(tf.reduce_sum(all_ex * gate, axis=1))
        return final_outputs

class ResNetBlock1D(Layer):
    def __init__(self, filters, kernel_size, strides=1, **kwargs):
        super().__init__(**kwargs)
        self.filters, self.kernel_size, self.strides = filters, kernel_size, strides
    def build(self, input_shape):
        self.conv1 = Conv1D(self.filters, self.kernel_size, strides=self.strides, padding='same', use_bias=False)
        self.bn1 = BatchNormalization()
        self.conv2 = Conv1D(self.filters, self.kernel_size, padding='same', use_bias=False)
        self.bn2 = BatchNormalization()
        self.shortcut = Conv1D(self.filters, 1, strides=self.strides, padding='same') if input_shape[-1] != self.filters or self.strides != 1 else Lambda(lambda x: x)
    def call(self, inputs):
        x = tf.nn.relu(self.bn1(self.conv1(inputs)))
        x = self.bn2(self.conv2(x))
        return tf.nn.relu(Add()([x, self.shortcut(inputs)]))

class EfficientTemporalEncoder(Layer):
    def __init__(self, **kwargs): super().__init__(**kwargs)
    def build(self, input_shape):
        self.conv = Conv1D(128, 3, padding='same', activation='relu')
        self.pool = GlobalAveragePooling1D()
    def call(self, inputs): return self.pool(self.conv(inputs))

# ---------------------- 3. 数据加载与适配 (核心修改) ----------------------
def load_dataset_aligned(root_dir):
    """直接读取制作好的 .npy 文件"""
    all_signals, all_labels, all_jnr_vals = [], [], []
    files = [f for f in os.listdir(root_dir) if f.endswith('_X.npy')]
    
    if not files:
        raise ValueError(f"❌ 在 {root_dir} 下未找到 _X.npy 文件！")

    print(f"📂 正在从 {len(files)} 个 .npy 文件中加载实测数据...")
    for fx in files:
        # 使用正则提取 JNR 数值
        match = re.search(r'jnr(-?\d+)', fx)
        jnr_val = float(match.group(1)) if match else 0.0
        
        signals = np.load(os.path.join(root_dir, fx))
        labels = np.load(os.path.join(root_dir, fx.replace('_X.npy', '_Y.npy')))
        
        # 统一形状：如果是 [N, 1024, 2]，取实部或压平
        if signals.ndim == 3: signals = signals[:, :, 0] 
        
        all_signals.append(signals)
        all_labels.append(labels)
        all_jnr_vals.append(np.full(len(labels), jnr_val))

    return {
        "signals": np.vstack(all_signals),
        "labels": np.concatenate(all_labels),
        "jnr_values": np.concatenate(all_jnr_vals),
        "fs": 200e3, "L": 1024
    }

def preprocess_data_aligned(dataset):
    """构建多任务标签：检测、分类、回归"""
    signals = dataset["signals"]
    labels = dataset["labels"]
    jnr_values = dataset["jnr_values"]
    
    # 归一化信号
    sig_norm = np.array([StandardScaler().fit_transform(s.reshape(-1, 1)).ravel() for s in signals])
    
    # 物理真值重建 (回归任务)
    bw_norm = np.array([TRUTH_MAP[l][0]/FS for l in labels])
    f0_norm = np.array([TRUTH_MAP[l][1]/(FS/2) for l in labels])
    jnr_lin = 10**(jnr_values / 10)
    jnr_norm = np.clip((jnr_lin - 10**-1)/(10**3 - 10**-1), 0, 1)
    
    params = np.stack([bw_norm, f0_norm, jnr_norm], axis=1).astype(np.float32)
    det_labels = (labels < 8).astype(np.float32) # 0-7是干扰，8是卫星

    return train_test_split(sig_norm, det_labels, labels, params, jnr_values, 
                            test_size=0.3, random_state=42, stratify=labels)

# ---------------------- 4. 模型构建与训练 ----------------------
def build_resnet_ple(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    # 时域分支
    t = ResNetBlock1D(64, 7, strides=2)(Reshape((input_shape[0], 1))(inputs))
    t = GlobalAveragePooling1D()(t)
    # 时序编码
    te = EfficientTemporalEncoder()(Reshape((input_shape[0], 1))(inputs))
    # 融合与PLE
    fusion = Concatenate()([t, te])
    ple = PLELayer(num_tasks=3, num_shared_experts=2, num_task_experts=1, expert_dim=128)(fusion)
    
    det = Dense(1, activation='sigmoid', name='detection_output')(ple[0])
    cls = Dense(num_classes, activation='softmax', name='classification_output')(ple[1])
    reg = Dense(3, activation='linear', name='reg_clip')(ple[2])
    
    model = Model(inputs, [det, cls, reg])
    model.compile(optimizer=Adam(1e-4), 
                  loss={'detection_output': 'binary_crossentropy', 'classification_output': 'sparse_categorical_crossentropy', 'reg_clip': 'mse'},
                  loss_weights={'detection_output': 1.0, 'classification_output': 2.0, 'reg_clip': 0.5})
    return model

def evaluate_and_report(model, X_test, y_det, y_type, y_param, j_test):
    """完整评估报表输出"""
    d, c, r = model.predict(X_test, batch_size=128)
    p_det = (d > 0.5).astype(int).ravel()
    p_cls = np.argmax(c, axis=1)
    
    print("\n" + "="*60)
    print("📊 快速ResNet-PLE收官评估报表")
    print("="*60)
    print(f"📡 干扰检测准确率: {accuracy_score(y_det, p_det)*100:.2f}%")
    print(f"🎯 总体识别准确率: {accuracy_score(y_type, p_cls)*100:.2f}%")
    
    print("\n📈 不同 JNR 下的识别率:")
    for j in sorted(np.unique(j_test)):
        mask = j_test == j
        print(f"   🔹 JNR {j:>3} dB: {accuracy_score(y_type[mask], p_cls[mask])*100:.2f}%")
        
    print("\n📝 干扰参数估计 (MAE & NRMSE):")
    p_names = ['带宽 (BW)', '中心频率 (Fc)', '干扰强度 (JNR)']
    maes = mean_absolute_error(y_param, r, multioutput='raw_values')
    nrmses = [np.sqrt(np.mean((y_param[:,i]-r[:,i])**2))/(np.max(y_param[:,i])-np.min(y_param[:,i])+1e-8) for i in range(3)]
    for n, m, nr in zip(p_names, maes, nrmses):
        print(f"   🔹 {n}: MAE = {m:.4f}, NRMSE = {nr:.4f}")
    print(f"🌟 平均预测 NRMSE: {np.mean(nrmses):.4f}")
    print("="*60)

# ---------------------- 5. 主流程 ----------------------
def main():
    root_path = "/root/autodl-tmp/validate/0218/dataset_final_ready_v5" 
    ds = load_dataset_aligned(root_path)
    X_train, X_test, y_det_t, y_det_v, y_type_t, y_type_v, y_param_t, y_param_v, j_t, j_v = preprocess_data_aligned(ds)
    
    model = build_resnet_ple((1024,), 9)
    print("🚀 开始训练...")
    model.fit(X_train, {'detection_output': y_det_t, 'classification_output': y_type_t, 'reg_clip': y_param_t},
              validation_split=0.1, epochs=30, batch_size=128, 
              callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
    
    evaluate_and_report(model, X_test, y_det_v, y_type_v, y_param_v, j_v)

if __name__ == "__main__":
    main()

📂 正在从 18 个 .npy 文件中加载实测数据...


2026-02-19 21:35:08.454257: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21164 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090 D, pci bus id: 0000:39:00.0, compute capability: 8.9


🚀 开始训练...
Epoch 1/30


2026-02-19 21:35:12.870413: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:424] Loaded cuDNN version 8600
2026-02-19 21:35:13.333871: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:637] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-02-19 21:35:13.414019: I tensorflow/compiler/xla/service/service.cc:169] XLA service 0x7fda2c01cc40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-19 21:35:13.414066: I tensorflow/compiler/xla/service/service.cc:177]   StreamExecutor device (0): NVIDIA GeForce RTX 4090 D, Compute Capability 8.9
2026-02-19 21:35:13.438705: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-19 21:35:13.666032: I ./tensorflow/compiler/jit/device_compiler.h:180] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the

670/670 [==============================] - 19s 18ms/step - loss: 3.8284 - detection_output_loss: 0.0656 - classification_output_loss: 1.8758 - reg_clip_loss: 0.0223 - val_loss: 3.4193 - val_detection_output_loss: 0.0012 - val_classification_output_loss: 1.7038 - val_reg_clip_loss: 0.0209
Epoch 2/30
670/670 [==============================] - 10s 16ms/step - loss: 3.1491 - detection_output_loss: 4.4549e-04 - classification_output_loss: 1.5699 - reg_clip_loss: 0.0177 - val_loss: 3.0112 - val_detection_output_loss: 1.3861e-04 - val_classification_output_loss: 1.5012 - val_reg_clip_loss: 0.0173
Epoch 3/30
670/670 [==============================] - 11s 16ms/step - loss: 2.8751 - detection_output_loss: 8.5040e-05 - classification_output_loss: 1.4336 - reg_clip_loss: 0.0156 - val_loss: 2.7972 - val_detection_output_loss: 6.5066e-05 - val_classification_output_loss: 1.3948 - val_reg_clip_loss: 0.0151
Epoch 4/30
670/670 [==============================] - 10s 15ms/step - loss: 2.7086 - detection_

In [6]:
# PLE_ResNet_fast_final_v5_Ensemble_Fixed.py
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                               recall_score, f1_score, mean_absolute_error)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, Conv2D, BatchNormalization, MaxPooling1D, MaxPooling2D,
                                     GlobalAveragePooling1D, GlobalAveragePooling2D, Dense, Reshape, Layer,
                                     Concatenate, Lambda, Add, Multiply, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import time

# ---------------------- 1. 物理参数定义 ----------------------
TYPE_NAMES = ["SingleTone", "Chirp", "Pulse", "HoppingJam", "NoiseFM", "NoiseAM", "Comb", "Mixed", "Normal"]
TRUTH_MAP = {
    0: [500, 0], 1: [60000, 0], 2: [45000, 0], 3: [85000, 0],
    4: [70000, 0], 5: [35000, 0], 6: [65000, 0], 7: [80000, 0], 8: [25000, 0]
}
FS = 200000

# ---------------------- 2. 自定义组件 (完全不动) ----------------------
class PLELayer(Layer):
    def __init__(self, num_tasks, num_shared_experts, num_task_experts, expert_dim, **kwargs):
        super(PLELayer, self).__init__(**kwargs)
        self.num_tasks, self.num_shared_experts, self.num_task_experts, self.expert_dim = num_tasks, num_shared_experts, num_task_experts, expert_dim
    def build(self, input_shape):
        self.shared_experts = [Dense(self.expert_dim, activation='relu') for _ in range(self.num_shared_experts)]
        self.task_experts = [[Dense(self.expert_dim, activation='relu') for _ in range(self.num_task_experts)] for _ in range(self.num_tasks)]
        self.gates = [Dense(self.num_shared_experts + self.num_task_experts, activation='softmax') for _ in range(self.num_tasks)]
    def call(self, inputs):
        shared_outputs = [ex(inputs) for ex in self.shared_experts]
        final_outputs = []
        for t in range(self.num_tasks):
            task_outputs = [ex(inputs) for ex in self.task_experts[t]]
            all_ex = tf.stack(shared_outputs + task_outputs, axis=1)
            gate = tf.expand_dims(self.gates[t](inputs), axis=-1)
            final_outputs.append(tf.reduce_sum(all_ex * gate, axis=1))
        return final_outputs

class ResNetBlock1D(Layer):
    def __init__(self, filters, kernel_size, strides=1, **kwargs):
        super(ResNetBlock1D, self).__init__(**kwargs)
        self.filters, self.kernel_size, self.strides = filters, kernel_size, strides
    def build(self, input_shape):
        self.conv1 = Conv1D(self.filters, self.kernel_size, strides=self.strides, padding='same', use_bias=False)
        self.bn1 = BatchNormalization()
        self.conv2 = Conv1D(self.filters, self.kernel_size, padding='same', use_bias=False)
        self.bn2 = BatchNormalization()
        self.shortcut = Conv1D(self.filters, 1, strides=self.strides, padding='same') if input_shape[-1] != self.filters or self.strides != 1 else Lambda(lambda x: x)
    def call(self, inputs):
        x = tf.nn.relu(self.bn1(self.conv1(inputs)))
        x = self.bn2(self.conv2(x))
        return tf.nn.relu(Add()([x, self.shortcut(inputs)]))

class EfficientTemporalEncoder(Layer):
    def __init__(self, **kwargs): super(EfficientTemporalEncoder, self).__init__(**kwargs)
    def build(self, input_shape):
        self.conv = Conv1D(128, 3, padding='same', activation='relu')
        self.pool = GlobalAveragePooling1D()
    def call(self, inputs): return self.pool(self.conv(inputs))

# ---------------------- 3. 数据加载与适配 (针对 v5 npy) ----------------------
def load_dataset_aligned(root_dir):
    """适配 v5 目录下的 .npy 文件加载"""
    all_signals, all_labels, all_jnr_vals = [], [], []
    files = [f for f in os.listdir(root_dir) if f.endswith('_X.npy')]
    if not files: raise ValueError(f"❌ 路径 {root_dir} 下未找到数据文件！")

    print(f"📂 正在加载 {len(files)} 个实测数据文件...")
    for fx in files:
        # 正则解析文件名中的 JNR 值
        match = re.search(r'jnr(-?\d+)', fx)
        jnr_val = float(match.group(1)) if match else 0.0
        
        sig = np.load(os.path.join(root_dir, fx))
        lab = np.load(os.path.join(root_dir, fx.replace('_X.npy', '_Y.npy')))
        
        # 统一形状：如果是 [N, 1024, 2]，提取实部
        if sig.ndim == 3: sig = sig[:, :, 0]
        
        all_signals.append(sig)
        all_labels.append(lab)
        all_jnr_vals.append(np.full(len(lab), jnr_val))

    return {
        "signals": np.vstack(all_signals),
        "labels": np.concatenate(all_labels),
        "jnr_values": np.concatenate(all_jnr_vals),
        "L": 1024, "fs": 200e3
    }

def preprocess_data_v5(dataset):
    """生成多任务标签并划分数据集"""
    sig = dataset["signals"]
    lab = dataset["labels"]
    jnr = dataset["jnr_values"]
    
    # 标准化
    sig_norm = np.array([StandardScaler().fit_transform(s.reshape(-1, 1)).ravel() for s in sig])
    
    # 参数回归真值重建
    bw_norm = np.array([TRUTH_MAP[l][0]/FS for l in lab])
    f0_norm = np.array([TRUTH_MAP[l][1]/(FS/2) for l in lab])
    # 归一化 JNR (线性映射到 0-1)
    jnr_norm = np.clip((10**(jnr/10) - 10**-1)/(10**3 - 10**-1), 0, 1)
    
    params = np.stack([bw_norm, f0_norm, jnr_norm], axis=1).astype(np.float32)
    det_lab = (lab < 8).astype(np.float32) # 0-7为干扰，8为卫星信号

    return train_test_split(sig_norm, det_lab, lab, params, jnr, 
                            test_size=0.3, random_state=42, stratify=lab)

# ---------------------- 4. 模型构建 (核心分支不动) ----------------------
def build_single_resnet_ple(L, num_classes):
    """
    单体模型构建
    修正了 shape=(L,) 的 TypeError 问题
    """
    inputs = Input(shape=(int(L),), name='input_signal') # 强制将 L 转为整数
    
    # --- 保持原有三分支逻辑不变 ---
    # 1. 时域 ResNet 分支
    t = ResNetBlock1D(64, 7, strides=2)(Reshape((L, 1))(inputs))
    t = GlobalAveragePooling1D()(t)
    
    # 2. 时序编码分支
    te = EfficientTemporalEncoder()(Reshape((L, 1))(inputs))
    
    # 3. 统计特征分支
    def stats_feat(x):
        return tf.concat([tf.reduce_mean(x, 1, True), tf.math.reduce_std(x, 1, True)], 1)
    st = Dense(64, activation='relu')(Lambda(stats_feat)(inputs))
    
    # 特征融合与 PLE 层
    fusion = Concatenate()([t, te, st])
    ple_out = PLELayer(3, 2, 1, 128)(fusion)
    
    # 多任务输出
    det = Dense(1, activation='sigmoid', name='detection_output')(ple_out[0])
    cls = Dense(num_classes, activation='softmax', name='classification_output')(ple_out[1])
    reg = Dense(3, activation='linear', name='reg_clip')(ple_out[2])
    
    model = Model(inputs, [det, cls, reg])
    model.compile(optimizer=Adam(1e-4), 
                  loss={'detection_output': 'binary_crossentropy', 'classification_output': 'sparse_categorical_crossentropy', 'reg_clip': 'mse'},
                  loss_weights={'detection_output': 1.0, 'classification_output': 2.0, 'reg_clip': 0.5})
    return model

# ---------------------- 5. 评估与指标分析 ----------------------
def evaluate_ensemble_v5(models, X_test, y_det, y_type, y_param, j_test):
    """集成评估逻辑"""
    all_d, all_c, all_r = [], [], []
    for m in models:
        d, c, r = m.predict(X_test, batch_size=128, verbose=0)
        all_d.append(d); all_c.append(c); all_r.append(r)
    
    # 集成平均预测
    p_det = (np.mean(all_d, axis=0) > 0.5).astype(int).ravel()
    p_cls = np.argmax(np.mean(all_c, axis=0), axis=1)
    p_reg = np.mean(all_r, axis=0)
    
    print("\n" + "="*60)
    print("📊 快速ResNet-PLE集成评估报表 (实测 v5 数据集)")
    print("="*60)
    print(f"📡 干扰检测准确率: {accuracy_score(y_det, p_det)*100:.2f}%")
    print(f"🎯 总体识别准确率: {accuracy_score(y_type, p_cls)*100:.2f}%")
    
    print("\n📈 不同 JNR 下的识别准确率:")
    for j in sorted(np.unique(j_test)):
        mask = j_test == j
        print(f"   🔹 JNR {j:>4} dB: {accuracy_score(y_type[mask], p_cls[mask])*100:.2f}%")
        
    print("\n📝 干扰参数估计 (MAE & NRMSE):")
    p_names = ['带宽 (BW)', '中心频率 (Fc)', '干扰强度 (JNR)']
    maes = mean_absolute_error(y_param, p_reg, multioutput='raw_values')
    
    nrmses = []
    for i in range(3):
        rmse = np.sqrt(np.mean((y_param[:,i] - p_reg[:,i])**2))
        data_range = np.max(y_param[:,i]) - np.min(y_param[:,i])
        nrmses.append(rmse / (data_range if data_range > 0 else 1.0)) # 保护除零
        
    for n, m, nr in zip(p_names, maes, nrmses):
        print(f"   🔹 {n}: MAE = {m:.4f}, NRMSE = {nr:.4f}")
    print(f"🌟 平均预测 NRMSE: {np.mean(nrmses):.4f}")
    print("="*60)

# ---------------------- 6. 主程序流程 ----------------------
def main():
    # 数据集路径对齐
    root_path = "/root/autodl-tmp/validate/0218/dataset_final_ready_v5" 
    os.makedirs("models", exist_ok=True)
    
    # 加载与预处理
    ds = load_dataset_aligned(root_path)
    X_train, X_test, y_det_t, y_det_v, y_type_t, y_type_v, y_param_t, y_param_v, j_t, j_v = preprocess_data_v5(ds)
    
    # 执行集成训练
    ensemble_models = []
    n_ensemble = 3 # 您最初设定的集成数量
    for i in range(n_ensemble):
        print(f"\n🔥 正在启动集成模型训练 [{i+1}/{n_ensemble}]...")
        # 传入 ds["L"] 时强制确保其为整数类型
        m = build_single_resnet_ple(int(ds["L"]), 9)
        m.fit(X_train, {'detection_output': y_det_t, 'classification_output': y_type_t, 'reg_clip': y_param_t},
              validation_split=0.1, epochs=30, batch_size=128, 
              callbacks=[EarlyStopping(patience=5, restore_best_weights=True)], verbose=1)
        ensemble_models.append(m)
    
    # 综合评估报表
    evaluate_ensemble_v5(ensemble_models, X_test, y_det_v, y_type_v, y_param_v, j_v)

if __name__ == "__main__":
    main()

📂 正在加载 18 个实测数据文件...



🔥 正在启动集成模型训练 [1/3]...
Epoch 1/30
670/670 [==============================] - 17s 18ms/step - loss: 3.8572 - detection_output_loss: 0.0551 - classification_output_loss: 1.8953 - reg_clip_loss: 0.0230 - val_loss: 3.4574 - val_detection_output_loss: 0.0016 - val_classification_output_loss: 1.7223 - val_reg_clip_loss: 0.0224
Epoch 2/30
670/670 [==============================] - 11s 16ms/step - loss: 3.2155 - detection_output_loss: 3.7276e-04 - classification_output_loss: 1.6029 - reg_clip_loss: 0.0188 - val_loss: 3.0738 - val_detection_output_loss: 1.0996e-04 - val_classification_output_loss: 1.5324 - val_reg_clip_loss: 0.0177
Epoch 3/30
670/670 [==============================] - 11s 16ms/step - loss: 2.9606 - detection_output_loss: 5.5127e-05 - classification_output_loss: 1.4763 - reg_clip_loss: 0.0161 - val_loss: 2.8660 - val_detection_output_loss: 2.8524e-05 - val_classification_output_loss: 1.4293 - val_reg_clip_loss: 0.0148
Epoch 4/30
670/670 [==============================] - 11s 16m

In [7]:
# PLE_ResNet_Ensemble_Final_v11.py
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, accuracy_score, mean_absolute_error)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, GlobalAveragePooling1D, 
                                     Dense, Reshape, Layer, Concatenate, Lambda, Add, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import time

# ---------------------- 1. 物理参数与科研指标定义 ----------------------
TYPE_NAMES = ["SingleTone", "Chirp", "Pulse", "HoppingJam", "NoiseFM", "NoiseAM", "Comb", "Mixed", "Normal"]
# 理论真值映射 (用于 MAE 对齐)
TRUTH_MAP = {
    0: [500, 0], 1: [60000, 0], 2: [45000, 0], 3: [85000, 0],
    4: [70000, 0], 5: [35000, 0], 6: [65000, 0], 7: [80000, 0], 8: [25000, 0]
}
FS = 200000

# ---------------------- 2. 核心 PLE 与 ResNet 分支组件 ----------------------
class PLELayer(Layer):
    """分层抽取共享专家特征的 PLE 层"""
    def __init__(self, num_tasks, num_shared_experts, num_task_experts, expert_dim, **kwargs):
        super(PLELayer, self).__init__(**kwargs)
        self.num_tasks, self.num_shared_experts, self.num_task_experts, self.expert_dim = num_tasks, num_shared_experts, num_task_experts, expert_dim
    def build(self, input_shape):
        self.shared_experts = [Dense(self.expert_dim, activation='relu') for _ in range(self.num_shared_experts)]
        self.task_experts = [[Dense(self.expert_dim, activation='relu') for _ in range(self.num_task_experts)] for _ in range(self.num_tasks)]
        self.gates = [Dense(self.num_shared_experts + self.num_task_experts, activation='softmax') for _ in range(self.num_tasks)]
    def call(self, inputs):
        shared_outputs = [ex(inputs) for ex in self.shared_experts]
        final_outputs = []
        for t in range(self.num_tasks):
            task_outputs = [ex(inputs) for ex in self.task_experts[t]]
            all_ex = tf.stack(shared_outputs + task_outputs, axis=1)
            gate = tf.expand_dims(self.gates[t](inputs), axis=-1)
            final_outputs.append(tf.reduce_sum(all_ex * gate, axis=1))
        return final_outputs

class ResNetBlock1D(Layer):
    """保持时域特征跳连的残差块"""
    def __init__(self, filters, kernel_size, strides=1, **kwargs):
        super(ResNetBlock1D, self).__init__(**kwargs)
        self.filters, self.kernel_size, self.strides = filters, kernel_size, strides
    def build(self, input_shape):
        self.conv1 = Conv1D(self.filters, self.kernel_size, strides=self.strides, padding='same', use_bias=False)
        self.bn1 = BatchNormalization()
        self.conv2 = Conv1D(self.filters, self.kernel_size, padding='same', use_bias=False)
        self.bn2 = BatchNormalization()
        # 修复快捷连接维度对齐逻辑
        self.shortcut = Conv1D(self.filters, 1, strides=self.strides, padding='same') if input_shape[-1] != self.filters or self.strides != 1 else Lambda(lambda x: x)
    def call(self, inputs):
        x = tf.nn.relu(self.bn1(self.conv1(inputs)))
        x = self.bn2(self.conv2(x))
        return tf.nn.relu(Add()([x, self.shortcut(inputs)]))

class EfficientTemporalEncoder(Layer):
    """用于捕获短时跳变特征的高效时序编码器"""
    def __init__(self, **kwargs): super(EfficientTemporalEncoder, self).__init__(**kwargs)
    def build(self, input_shape):
        self.conv = Conv1D(128, 3, padding='same', activation='relu')
        self.pool = GlobalAveragePooling1D()
    def call(self, inputs): return self.pool(self.conv(inputs))

# ---------------------- 3. 数据加载与预处理 (适配 v5 npy) ----------------------
def load_dataset_aligned(root_dir):
    """直接读取 v5 版本 .npy 文件并自动解析 JNR"""
    all_signals, all_labels, all_jnr_vals = [], [], []
    files = [f for f in os.listdir(root_dir) if f.endswith('_X.npy')]
    if not files: raise ValueError(f"❌ 路径 {root_dir} 下未找到有效数据！")

    print(f"📂 正在读取 {len(files)} 个数据文件并应用物理隔离切片...")
    for fx in files:
        # 兼容正负 JNR 值的正则表达式
        match = re.search(r'jnr(-?\d+)', fx)
        jnr_val = float(match.group(1)) if match else 0.0
        sig = np.load(os.path.join(root_dir, fx))
        lab = np.load(os.path.join(root_dir, fx.replace('_X.npy', '_Y.npy')))
        # 转换 [Batch, 1024, 2] 为 [Batch, 1024] (仅实部)
        if sig.ndim == 3: sig = sig[:, :, 0] 
        all_signals.append(sig); all_labels.append(lab); all_jnr_vals.append(np.full(len(lab), jnr_val))

    return {"signals": np.vstack(all_signals), "labels": np.concatenate(all_labels), "jnr_values": np.concatenate(all_jnr_vals), "L": 1024}

def preprocess_v5(dataset):
    """将实测数据转换为三任务标签"""
    sig, lab, jnr = dataset["signals"], dataset["labels"], dataset["jnr_values"]
    sig_norm = np.array([StandardScaler().fit_transform(s.reshape(-1, 1)).ravel() for s in sig])
    
    # 物理参数归一化 (带宽/FS, 频偏/(FS/2), JNR线性映射)
    bw_norm = np.array([TRUTH_MAP[l][0]/FS for l in lab])
    f0_norm = np.array([TRUTH_MAP[l][1]/(FS/2) for l in lab])
    jnr_norm = np.clip((10**(jnr/10) - 0.1)/(1000 - 0.1), 0, 1)
    
    params = np.stack([bw_norm, f0_norm, jnr_norm], axis=1).astype(np.float32)
    det_lab = (lab < 8).astype(np.float32) # 0-7是干扰，8是卫星信号

    return train_test_split(sig_norm, det_lab, lab, params, jnr, test_size=0.3, random_state=42, stratify=lab)

# ---------------------- 4. 核心模型构建 (修复 TypeError) ----------------------
def build_single_resnet_ple(L, num_classes):
    """修复了维度传递错误的单体多任务模型"""
    # 修复 Input shape 定义
    inputs = Input(shape=(L,), name='input_signal')
    
    # 1. 时域 ResNet 分支
    # 确保 Reshape 接收的是整数维
    t = ResNetBlock1D(64, 7, strides=2)(Reshape((L, 1))(inputs))
    t = GlobalAveragePooling1D()(t)
    
    # 2. 时序编码分支
    te = EfficientTemporalEncoder()(Reshape((L, 1))(inputs))
    
    # 3. 统计特征分支
    def statistical_calc(x):
        return tf.concat([tf.reduce_mean(x, 1, True), tf.math.reduce_std(x, 1, True)], 1)
    st = Dense(64, activation='relu')(Lambda(statistical_calc)(inputs))
    
    # 特征融合层
    fusion = Concatenate()([t, te, st])
    ple_out = PLELayer(3, 2, 1, 128)(fusion)
    
    # 三任务头：检测、分类、回归
    det = Dense(1, activation='sigmoid', name='detection_output')(ple_out[0])
    cls = Dense(num_classes, activation='softmax', name='classification_output')(ple_out[1])
    reg = Dense(3, activation='linear', name='reg_clip')(ple_out[2])
    
    model = Model(inputs, [det, cls, reg])
    model.compile(optimizer=Adam(1e-4), 
                  loss={'detection_output': 'binary_crossentropy', 'classification_output': 'sparse_categorical_crossentropy', 'reg_clip': 'mse'},
                  loss_weights={'detection_output': 1.0, 'classification_output': 5.0, 'reg_clip': 0.5})
    return model

# ---------------------- 5. 进度统计与多指标评估 ----------------------
def evaluate_and_progress(models, X_test, y_det, y_type, y_param, j_test):
    """集成模型评估与物理量误差统计"""
    preds_d, preds_c, preds_r = [], [], []
    for m in models:
        d, c, r = m.predict(X_test, batch_size=128, verbose=0)
        preds_d.append(d); preds_c.append(c); preds_r.append(r)
    
    # 集成平均
    final_det = (np.mean(preds_d, 0) > 0.5).astype(int).ravel()
    final_cls = np.argmax(np.mean(preds_c, 0), axis=1)
    final_reg = np.mean(preds_r, 0)
    
    print("\n" + "="*60)
    print("📊 实测全维度科研评估报表 (Ensemble Version)")
    print("="*60)
    print(f"📡 干扰检测准确率: {accuracy_score(y_det, final_det)*100:.2f}%")
    print(f"🎯 干扰识别准确率: {accuracy_score(y_type, final_cls)*100:.2f}%")
    
    print("\n📈 [进度] 不同 JNR 档位识别性能统计:")
    for j in sorted(np.unique(j_test)):
        mask = (j_test == j)
        acc = accuracy_score(y_type[mask], final_cls[mask])
        print(f"   🔹 JNR {j:>3} dB: {acc*100:6.2f}% 完成度 [OK]")
        
    print("\n📝 [进度] 干扰参数回归 MAE & NRMSE 误差统计:")
    p_names = ['带宽 (BW)', '中心频率 (Fc)', '干扰强度 (JNR)']
    maes = mean_absolute_error(y_param, final_reg, multioutput='raw_values')
    nrmses = []
    for i in range(3):
        rmse = np.sqrt(np.mean((y_param[:,i] - final_reg[:,i])**2))
        nrmses.append(rmse / (np.max(y_param[:,i]) - np.min(y_param[:,i]) + 1e-8))
        
    for n, m, nr in zip(p_names, maes, nrmses):
        print(f"   🔹 {n}: MAE={m:.4f}, NRMSE={nr:.4f} [对齐成功]")
    print(f"🌟 平均预测误差 (Total NRMSE): {np.mean(nrmses):.4f}")
    print("="*60)

# ---------------------- 6. 主程序入口 ----------------------
def main():
    root_path = "/root/autodl-tmp/validate/0218/dataset_final_ready_v5" 
    
    # 1. 加载与预处理
    ds = load_dataset_aligned(root_path)
    X_train, X_test, y_det_t, y_det_v, y_type_t, y_type_v, y_param_t, y_param_v, j_t, j_v = preprocess_v5(ds)
    
    # 2. 训练集成模型 (n_models=3)
    ensemble_list = []
    for i in range(3):
        print(f"\n🔥 启动第 {i+1}/3 个集成模型的训练进程...")
        m = build_single_resnet_ple(1024, 9)
        m.fit(X_train, {'detection_output': y_det_t, 'classification_output': y_type_t, 'reg_clip': y_param_t},
              validation_split=0.1, epochs=30, batch_size=128, 
              callbacks=[EarlyStopping(patience=5, restore_best_weights=True)], verbose=1)
        ensemble_list.append(m)
    
    # 3. 最终收官评估
    evaluate_and_progress(ensemble_list, X_test, y_det_v, y_type_v, y_param_v, j_v)

if __name__ == "__main__":
    main()

📂 正在读取 18 个数据文件并应用物理隔离切片...

🔥 启动第 1/3 个集成模型的训练进程...
Epoch 1/30
670/670 [==============================] - 18s 17ms/step - loss: 9.5174 - detection_output_loss: 0.0759 - classification_output_loss: 1.8861 - reg_clip_loss: 0.0220 - val_loss: 8.5282 - val_detection_output_loss: 0.0012 - val_classification_output_loss: 1.7033 - val_reg_clip_loss: 0.0212
Epoch 2/30
670/670 [==============================] - 11s 17ms/step - loss: 7.8377 - detection_output_loss: 4.7988e-04 - classification_output_loss: 1.5657 - reg_clip_loss: 0.0172 - val_loss: 7.4739 - val_detection_output_loss: 1.5492e-04 - val_classification_output_loss: 1.4931 - val_reg_clip_loss: 0.0168
Epoch 3/30
670/670 [==============================] - 11s 17ms/step - loss: 7.1877 - detection_output_loss: 6.1245e-05 - classification_output_loss: 1.4360 - reg_clip_loss: 0.0153 - val_loss: 6.9532 - val_detection_output_loss: 3.3398e-05 - val_classification_output_loss: 1.3892 - val_reg_clip_loss: 0.0143
Epoch 4/30
670/670 [===========